In [1]:
# BVP - Batter vs. Pitcher Stats Example



import statsapi
import mlbstatsapi

mlb = mlbstatsapi.Mlb()
batter_id = mlb.get_people_id('Pete Alonso')[0]
pitcher_id = mlb.get_people_id('Griffin Canning')[0]

stats = ['vsPlayer']
group = ['hitting']
params = {'opposingPlayerId': pitcher_id, 'season': 2025}

stats = mlb.get_player_stats(batter_id, stats=stats, groups=group, **params)
vs_player_total = stats['hitting']['vsplayertotal']
for split in vs_player_total.splits:
    for k, v in split.stat.__dict__.items():
            print(k, v)


https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/people/624413/stats
gamesplayed 2
flyouts None
groundouts 2
airouts 1
runs None
doubles 0
triples 0
homeruns 0
strikeouts 2
baseonballs 0
intentionalwalks 0
hits 1
hitbypitch 0
avg .167
atbats 6
obp .167
slg .167
ops .334
caughtstealing None
stolenbases None
stolenbasepercentage None
groundintodoubleplay 1
groundintotripleplay 0
numberofpitches 20
plateappearances 6
totalbases 1
rbi 0
leftonbase 6
sacbunts 0
sacflies 0
babip .250
groundoutstoairouts 2.00
catchersinterference 0
atbatsperhomerun -.--


In [ ]:
def get_matches_today_data():
    import statsapi
    import mlbstatsapi
    from datetime import datetime

    # Get today's schedule
    matches_today = []

    # get the proper formatted date
    mlb_date = datetime.now().strftime("%m/%d/%Y")

    # get the schedule as a dictionary for today
    schedule = statsapi.schedule(start_date=mlb_date, end_date=mlb_date)

    # iterate through each game of the schedule
    for x in schedule:

        # initialize game data dictionary
        game_data = {}
        
        # away_name
        game_data.update({'away_name': x.get('away_name')}) 
        
        # home_name
        game_data.update({'home_name': x.get('home_name')})
        
        # away_id
        game_data.update({'away_id': x.get('away_id')})

        away_team_leaders_hr = []
        # add top away team guys here
        away_leaders = statsapi.team_leader_data(x.get('away_id'), 'homeRuns', season=2025, leaderGameTypes="R", limit=10)
        for z in away_leaders:
            away_team_leaders_hr.append({'name': z[1],'homeRuns': z[2]})

        game_data.update({'away_team_leaders_hr': away_team_leaders_hr})

        home_team_leaders_hr = []
        # add top away team guys here
        home_leaders = statsapi.team_leader_data(x.get('home_id'), 'homeRuns', season=2025, leaderGameTypes="R", limit=10)
        for z in home_leaders:
            home_team_leaders_hr.append({'name': z[1],'homeRuns': z[2]})

        game_data.update({'home_team_leaders_hr': home_team_leaders_hr})

        # home_id
        game_data.update({'home_id': x.get('home_id')})
        # home_probable_pitcher
        game_data.update({'home_probable_pitcher': x.get('home_probable_pitcher')})
        # away_probable_pitcher
        game_data.update({'away_probable_pitcher': x.get('away_probable_pitcher')})

        matches_today.append(game_data)


    mlb = mlbstatsapi.Mlb()

    for x in matches_today:
  
        away_probable_pitcher = x.get('away_probable_pitcher')
        
        # Check if away_probable_pitcher is valid
        if not away_probable_pitcher:
            print(f"Warning: Missing away_probable_pitcher for game: {x}")
            continue  # Skip this game if no pitcher is available

        pitcher_ids = mlb.get_people_id(away_probable_pitcher)
        
        # Check if pitcher_ids is not empty
        if not pitcher_ids:
            print(f"Warning: No pitcher ID found for {away_probable_pitcher}")
            continue  # Skip this game if no pitcher ID is found

        pitcher_id = pitcher_ids[0]  # Safely access the first element

        BvP = []
        for y in x.get('home_team_leaders_hr', []):  # Default to an empty list if key is missing
            batter_id = mlb.get_people_id(y.get('name'))[0]

            stats = ['vsPlayer']
            group = ['hitting']
            params = {'opposingPlayerId': pitcher_id, 'season': 2025}

            try:
                stats = mlb.get_player_stats(batter_id, stats=stats, groups=group, **params)
                vs_player_total = stats['hitting']['vsplayertotal']
                for split in vs_player_total.splits:
                    p_id = mlb.get_person(pitcher_id)
                    b_id = mlb.get_person(batter_id)
                    
                    bvp_matchup = f"pitcher: {p_id.__dict__.get('fullname')} vs batter: {b_id.__dict__.get('fullname')}"
                    dict2 = {'bvp_stats': split.stat.__dict__}
                    dict2.update({'bvp_matchup': bvp_matchup})
                    BvP.append(dict2)

            except KeyError as e:
                print(f"KeyError: {e}. Skipping this player. Stats: {stats}")
            except Exception as e:
                print(f"Unexpected error: {e}. Skipping this player.")
        
        # Add the BvP stats to the matches_today dictionary
        x.update({'BvP_stats': BvP})

    return matches_today

# todays_matches = get_matches_today_data()

import sys
import os

def suppress_output():
    sys.stdout = open(os.devnull, 'w')

def restore_output():
    sys.stdout = sys.__stdout__

# Suppress output
suppress_output()

# Call your function
todays_matches = get_matches_today_data()

# Restore output
restore_output()

# Print a readable output of each item in the dictionary of the list matches_today
for match in todays_matches:
    print(f"Match: AWAY: {match['away_name']} vs HOME: {match['home_name']}")
    print(f"Away Team ID: {match['away_id']}, Home Team ID: {match['home_id']}")
    print(f"Away Probable Pitcher: {match['away_probable_pitcher']}")
    print(f"Home Probable Pitcher: {match['home_probable_pitcher']}")
    
    print("Away Team Leaders in Home Runs:")
    for leader in match['away_team_leaders_hr']:
        print(f"  {leader['name']}: {leader['homeRuns']} HR")
    
    print("Home Team Leaders in Home Runs:")
    for leader in match['home_team_leaders_hr']:
        print(f"  {leader['name']}: {leader['homeRuns']} HR")
    
    if 'BvP_stats' in match:
        print("Batter vs Pitcher Stats:")
        for bvp in match['BvP_stats']:
            print(f"  Matchup: {bvp['bvp_matchup']}")
            for stat, value in bvp['bvp_stats'].items():
                print(f"    {stat}: {value}")
    
    print("\n")  # Print a newline for better readability between matches

In [2]:


def get_matches_today_data():
    import statsapi
    import mlbstatsapi
    from datetime import datetime

    # Get today's schedule
    matches_today = []

    # get the proper formatted date
    mlb_date = datetime.now().strftime("%m/%d/%Y")

    # get the schedule as a dictionary for today
    schedule = statsapi.schedule(start_date=mlb_date, end_date=mlb_date)

    # iterate through each game of the schedule
    for x in schedule:

        # initialize game data dictionary
        game_data = {}
        
        # away_name
        game_data.update({'away_name': x.get('away_name')}) 
        
        # home_name
        game_data.update({'home_name': x.get('home_name')})
        
        # away_id
        game_data.update({'away_id': x.get('away_id')})

        away_team_leaders_hr = []
        # add top away team guys here
        away_leaders = statsapi.team_leader_data(x.get('away_id'), 'homeRuns', season=2025, leaderGameTypes="R", limit=10)
        for z in away_leaders:
            away_team_leaders_hr.append({'name': z[1],'homeRuns': z[2]})

        game_data.update({'away_team_leaders_hr': away_team_leaders_hr})

        home_team_leaders_hr = []
        # add top away team guys here
        home_leaders = statsapi.team_leader_data(x.get('home_id'), 'homeRuns', season=2025, leaderGameTypes="R", limit=10)
        for z in home_leaders:
            home_team_leaders_hr.append({'name': z[1],'homeRuns': z[2]})

        game_data.update({'home_team_leaders_hr': home_team_leaders_hr})

        # home_id
        game_data.update({'home_id': x.get('home_id')})
        # home_probable_pitcher
        game_data.update({'home_probable_pitcher': x.get('home_probable_pitcher')})
        # away_probable_pitcher
        game_data.update({'away_probable_pitcher': x.get('away_probable_pitcher')})

        matches_today.append(game_data)


    mlb = mlbstatsapi.Mlb()

    for x in matches_today:
        pitcher_id = mlb.get_people_id(x.get('away_probable_pitcher'))[0]
        
        BvP = []
        for y in x.get('home_team_leaders_hr'):
            batter_id = mlb.get_people_id(y.get('name'))[0]

            stats = ['vsPlayer']
            group = ['hitting']
            params = {'opposingPlayerId': pitcher_id, 'season': 2025}

            # stats = mlb.get_player_stats(batter_id, stats=stats, groups=group, **params)
            # vs_player_total = stats['hitting']['vsplayertotal']
            # for split in vs_player_total.splits:
            #     for k, v in split.stat.__dict__.items():
            #             print(k, v)
            try:
                stats = mlb.get_player_stats(batter_id, stats=stats, groups=group, **params)
                # Attempt to access 'hitting' key
                vs_player_total = stats['hitting']['vsplayertotal']
                for split in vs_player_total.splits:
                    # for k, v in split.stat.__dict__.items():
                    #     print(k, v)
                    # print(split)
                    # print(split.stat.__dict__.items())
                    p_id = mlb.get_person(pitcher_id)
                    b_id = mlb.get_person(batter_id)
                    
                    # add this info to the dictionary for matches to print a formatted copy later
                    bvp_matchup = f'pitcher: {p_id.__dict__.get('fullname')} vs batter: {b_id.__dict__.get('fullname')}'
                    dict2 = {'bvp_stats': split.stat.__dict__}
                    dict2.update({'bvp_matchup': bvp_matchup})
                    BvP.append(dict2)


            except KeyError as e:
                # Handle missing 'hitting' key or other KeyErrors
                #print(f"KeyError: {e}. Skipping this player. Stats: {stats}")
                yellow = 1
            except Exception as e:
                # Handle any other unexpected errors
                #print(f"Unexpected error: {e}. Skipping this player.")
                yellow = 2
        # add the BvP stats to the matches_today dictionary
        x.update({'BvP_stats': BvP})

    return matches_today

todays_matches = get_matches_today_data()


https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/people/682829/stats
https://statsapi.mlb.com/api/v1/people/571945
https://statsapi.mlb.com/api/v1/people/682829
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/people/669720/stats
https://statsapi.mlb.com/api/v1/people/571945
https://statsapi.mlb.com/api/v1/people/669720
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/people/680574/stats
https://statsapi.mlb.com/api/v1/people/571945
https://statsapi.mlb.com/api/v1/people/680574
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/people/682622/stats
https://statsapi.mlb.com/api/v1/people/571945
https://statsapi.mlb.com/api/v1/people/682622
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/people/642851/stats
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/

IndexError: list index out of range

In [7]:
# Print a readable output of each item in the dictionary of the list matches_today
for match in todays_matches:
    print(f"Match: AWAY: {match['away_name']} vs HOME: {match['home_name']}")
    print(f"Away Team ID: {match['away_id']}, Home Team ID: {match['home_id']}")
    print(f"Away Probable Pitcher: {match['away_probable_pitcher']}")
    print(f"Home Probable Pitcher: {match['home_probable_pitcher']}")
    
    print("Away Team Leaders in Home Runs:")
    for leader in match['away_team_leaders_hr']:
        print(f"  {leader['name']}: {leader['homeRuns']} HR")
    
    print("Home Team Leaders in Home Runs:")
    for leader in match['home_team_leaders_hr']:
        print(f"  {leader['name']}: {leader['homeRuns']} HR")
    
    if 'BvP_stats' in match:
        print("Batter vs Pitcher Stats:")
        for bvp in match['BvP_stats']:
            print(f"  Matchup: {bvp['bvp_matchup']}")
            for stat, value in bvp['bvp_stats'].items():
                print(f"    {stat}: {value}")
    
    print("\n")  # Print a newline for better readability between matches

In [15]:
for x in matches_today:
    print(f"Game: {x['away_name']} at {x['home_name']}")
    print(f"Away Team ID: {x['away_id']}, Home Team ID: {x['home_id']}")
    print(f"Away Probable Pitcher ID: {x['away_probable_pitcher']}, Home Probable Pitcher ID: {x['home_probable_pitcher']}")
    
    print("Away Team Leaders in Home Runs:")
    for leader in x['away_team_leaders_hr']:
        print(f"  {leader['name']}: {leader['homeRuns']} HR")
    
    print("Home Team Leaders in Home Runs:")
    for leader in x['home_team_leaders_hr']:
        print(f"  {leader['name']}: {leader['homeRuns']} HR")
    
    print("\n")  # Print a newline for better readability between games

Game: New York Mets at Washington Nationals
Away Team ID: 121, Home Team ID: 120
Away Probable Pitcher ID: Griffin Canning, Home Probable Pitcher ID: Trevor Williams
Away Team Leaders in Home Runs:
  Pete Alonso: 6 HR
  Francisco Lindor: 5 HR
  Brandon Nimmo: 4 HR
  Juan Soto: 3 HR
  Mark Vientos: 2 HR
  Francisco Alvarez: 1 HR
  Brett Baty: 1 HR
  Starling Marte: 1 HR
  Luis Torrens: 1 HR
  Jesse Winker: 1 HR
Home Team Leaders in Home Runs:
  James Wood: 8 HR
  CJ Abrams: 4 HR
  Josh Bell: 4 HR
  Dylan Crews: 4 HR
  Nathaniel Lowe: 4 HR
  Keibert Ruiz: 2 HR
  Riley Adams: 1 HR
  Luis García Jr.: 1 HR
  Amed Rosario: 1 HR


Game: Minnesota Twins at Cleveland Guardians
Away Team ID: 142, Home Team ID: 114
Away Probable Pitcher ID: Bailey Ober, Home Probable Pitcher ID: Gavin Williams
Away Team Leaders in Home Runs:
  Byron Buxton: 6 HR
  Trevor Larnach: 4 HR
  Harrison Bader: 3 HR
  Ty France: 2 HR
  Willi Castro: 1 HR
  Carlos Correa: 1 HR
  Edouard Julien: 1 HR
  Brooks Lee: 1 HR
  Jo

# Games today

## YESTERDAY RESULTS
[link to GAME_REPORT]

## STANDINGS

## TODAYS SCHEDULE

## TOP 7 LEAGUE_LEADERS

## Game #X

HOME [name] vs AWAY [name]

HOME [team leaders ['homeRuns']
AWAY [team leaders ['homeRuns']

HOME [probable pitcher]
GET top 7 AWAY leaders and compare stats against pitcher

AWAY [probable pitcher]
GET top 7 HOME leaders and compare stats against pitcher









